# 11.1 索引构建本节涵盖：稠密检索, 稀疏检索(BM25), 混合检索, 向量数据库

In [ ]:
import torchimport torch.nn as nnimport torch.nn.functional as Ffrom collections import defaultdicttorch.manual_seed(42)class DenseRetriever(nn.Module):    def __init__(self, d_model=64):        super().__init__()        self.query_encoder = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 16))        self.doc_encoder = nn.Sequential(nn.Linear(d_model, 32), nn.ReLU(), nn.Linear(32, 16))    def encode_query(self, q):        return F.normalize(self.query_encoder(q), dim=-1)    def encode_doc(self, d):        return F.normalize(self.doc_encoder(d), dim=-1)retriever = DenseRetriever()n_docs = 100queries = torch.randn(5, 64)docs = torch.randn(n_docs, 64)q_embeds = retriever.encode_query(queries)d_embeds = retriever.encode_doc(docs)scores = q_embeds @ d_embeds.Ttop_k = 5top_indices = scores.topk(top_k, dim=-1).indicesprint('=== Dense Retrieval ===')print(f'Queries: {queries.shape}, Documents: {docs.shape}')print(f'Query embeddings: {q_embeds.shape}, Doc embeddings: {d_embeds.shape}')print(f'Top-{top_k} indices for query 0: {top_indices[0].tolist()}')print()print('=== BM25 (Sparse Retrieval) ===')docs_text = ['machine learning algorithms', 'deep neural networks', 'natural language processing']query_text = 'neural network deep learning'def simple_bm25(query, docs, k1=1.5, b=0.75):    query_terms = query.lower().split()    scores = []    avgdl = sum(len(d.split()) for d in docs) / len(docs)    for doc in docs:        doc_terms = doc.lower().split()        dl = len(doc_terms)        score = 0        for term in query_terms:            tf = doc_terms.count(term)            df = sum(1 for d in docs if term in d.lower().split())            idf = math.log((len(docs) - df + 0.5) / (df + 0.5) + 1)            score += idf * (tf * (k1 + 1)) / (tf + k1 * (1 - b + b * dl / avgdl))        scores.append(score)    return scoresimport mathbm25_scores = simple_bm25(query_text, docs_text)print(f'Query: "{query_text}"')for doc, score in zip(docs_text, bm25_scores):    print(f'  Score={score:.3f}: "{doc}"')print('\n=== Hybrid Retrieval ===')print('Combine dense + sparse: RRF (Reciprocal Rank Fusion)')print('score = sum(1 / (k + rank_i)) for each retriever i')print('\n=== Vector Databases ===')print('Milvus, Pinecone, Weaviate, FAISS, ChromaDB')print('Support: ANN search, billion-scale vectors, real-time updates')